# Milestone 3

In [ ]:
!pip install faiss-cpu

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity




In [ ]:
#from google.colab import files
#upload = files.upload()

In [ ]:
train = pd.read_csv('/content/train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings.astype('float32'))

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base successfully created


---
## Zero-shot classifier for Q1, Q2, Q6

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

---
## Q1

Run the zero-shot classifier on `facebook/bart-large-mnli` on the prompt for the row index 150. Pass the 5 options (A–E) as `candidate_labels`. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [ ]:
result_q1 = zs(prompt_150, candidate_labels=labels_150)

label_to_score = dict(zip(result_q1['labels'], result_q1['scores']))
correct_score_q1 = label_to_score[ans_150]

print("All labels and scores:")
for label, score in zip(result_q1['labels'], result_q1['scores']):
    marker = " <-- CORRECT" if label == ans_150 else ""
    print(f"  {score:.4f}  {label}{marker}")

print(f"Ans: {round(correct_score_q1, 3)}")

All labels and scores:
  0.3844  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." <-- CORRECT
  0.3787  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
  0.0927  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."
  0.0786  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence 

---
## Q2

Embed the prompt for row index 150 using `all-MiniLM-L6-v2`. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [ ]:
prompt_150_emb = model.encode([prompt_150]).astype('float32')

distances, retrieved_indices = index.search(prompt_150_emb, 10)
retrieved_indices = retrieved_indices[0].tolist()

print(f"Top 10 retrieved KB indices: {retrieved_indices}")

if 150 in retrieved_indices:
    rank = retrieved_indices.index(150) + 1
    print(f"\nTrue document found at rank {rank}")
else:
    print("True document NOT in top 10")

Top 10 retrieved KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]

True document found at rank 10


---
## Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)

In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses a Bi-encoder.

A **Bi-encoder** embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allow you to search millions of documents but often miss semantic context.

To fix this we use a **2 stage pipeline**:
1. **Retrieval:** Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. **Reranking:** Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A **Cross-Encoder** passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

### Cross Encoders (Hugging Face)

Code to use a Cross-Encoder:
```python
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices]  # Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10]  # Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs)  # Get the score of each pair
```

---
## Q3

Take the top 10 documents retrieved by FAISS in the previous question. Load `cross-encoder/ms-marco-MiniLM-L-6-v2`. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

ranked_order = np.argsort(ce_scores)[::-1].tolist()
ranked_kb_indices = [retrieved_indices[i] for i in ranked_order]

print("Cross-Encoder ranked KB indices (rank 1 = best):")
for rank, (kb_idx, score_idx) in enumerate(zip(ranked_kb_indices, ranked_order), 1):
    marker = "TRUE DOC" if kb_idx == 150 else ""
    print(f"  Rank {rank}: KB index {kb_idx} | CE score {ce_scores[score_idx]:.4f}{marker}")

if 150 in ranked_kb_indices:
    ce_rank = ranked_kb_indices.index(150) + 1
    print(f"\nTrue document at rank {ce_rank} after Cross-Encoder reranking")
else:
    print("\nTrue document NOT in reranked top 10")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder ranked KB indices (rank 1 = best):
  Rank 1: KB index 150 | CE score 4.7585TRUE DOC
  Rank 2: KB index 1906 | CE score 4.7526
  Rank 3: KB index 847 | CE score 4.7526
  Rank 4: KB index 1693 | CE score 4.7526
  Rank 5: KB index 1269 | CE score 4.7375
  Rank 6: KB index 1532 | CE score 4.7375
  Rank 7: KB index 168 | CE score 4.7072
  Rank 8: KB index 576 | CE score 4.6870
  Rank 9: KB index 1701 | CE score 4.6602
  Rank 10: KB index 663 | CE score 4.6602

True document at rank 1 after Cross-Encoder reranking


---
## Q4

Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: `"Context: [concatenated_docs] Question: [prompt]"`. Tokenize this string using the `bert-base-uncased` tokenizer (without truncation). Exactly how many total tokens does this generate?

In [ ]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

prompt_42_emb = model.encode([prompt_42]).astype('float32')
_, retrieved_42 = index.search(prompt_42_emb, 5)
docs_5 = [kb[j] for j in retrieved_42[0]]

concatenated = ' '.join(docs_5)
rag_string_42 = f"Context: {concatenated} Question: {prompt_42}"

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens = bert_tokenizer(rag_string_42, truncation=False)
token_count = len(tokens['input_ids'])

print(f"Total tokens = {token_count}")

Total tokens = 216


---
## Q5

Retrieve the exact true document for row index 150 from your KB. Create a RAG string: `"Context: [true_document] Question: [prompt]"`. Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [ ]:
true_doc_150 = kb[150]
rag_string_q5 = f"Context: {true_doc_150} Question: {prompt_150}"

result_q5 = zs(rag_string_q5, candidate_labels=labels_150)

label_to_score_q5 = dict(zip(result_q5['labels'], result_q5['scores']))
correct_score_q5 = label_to_score_q5[ans_150]

print("All labels and scores (with true RAG context):")
for label, score in zip(result_q5['labels'], result_q5['scores']):
    marker = "CORRECT" if label == ans_150 else ""
    print(f"  {score:.4f}  {label}{marker}")

print(f"Ans: {round(correct_score_q5, 3)}")

All labels and scores (with true RAG context):
  0.9894  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."CORRECT
  0.0045  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
  0.0028  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
  0.0017  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book

---
## Concept: The Danger of Bad Retrieval (Adversarial RAG)

The golden rule of Retrieval-Augmented Generation is **"Garbage In, Garbage Out."** An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constraining the number of chunks that we give to the model for the same reason.

---
## Q6

What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [ ]:
wrong_doc = kb[999]
rag_string_q6 = f"Context: {wrong_doc} Question: {prompt_150}"

result_q6 = zs(rag_string_q6, candidate_labels=labels_150)

label_to_score_q6 = dict(zip(result_q6['labels'], result_q6['scores']))
correct_score_q6 = label_to_score_q6[ans_150]

print("All labels and scores (adversarial RAG):")
for label, score in zip(result_q6['labels'], result_q6['scores']):
    marker = "CORRECT" if label == ans_150 else ""
    print(f"{score:.4f} {label}{marker}")

print(f"\nQ6 ANSWER: {round(correct_score_q6, 3)}")
print(f"\nComparison:")
print(f"Q1(no RAG):{round(correct_score_q1, 3)}")
print(f"Q5(true doc): {round(correct_score_q5, 3)}")
print(f"Q6(wrong doc): {round(correct_score_q6, 3)}")

All labels and scores (adversarial RAG):
0.5289 The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."CORRECT
0.4253 The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.0204 The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.0187 The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.0067 The butterfly effect is the phenomenon that a small change in t

---
## Q7

For the first 100 rows of `train.csv` (indices 0–99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [ ]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    correct_text = str(row[row['answer']])

    prompt_emb = model.encode([prompt]).astype('float32')
    _, indices_5 = index.search(prompt_emb, 5)
    docs_5 = [kb[j] for j in indices_5[0]]

    if any(correct_text in doc for doc in docs_5):
        hits += 1

hit_rate = (hits / 100) * 100
print(f"Hits: {hits}/100")
print(f"Hit Rate = {round(hit_rate, 1)}%")

Hits: 73/100
Hit Rate = 73.0%


---
## Q8

Build a loop that processes the first 20 rows (indices 0 through 19) of `train.csv`. For each row, your pipeline must do the following in order:

1. **Retrieve:** Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.
2. **Rerank:** Pass the prompt and those 5 documents into the `ms-marco-MiniLM-L-6-v2` Cross-Encoder. Select the single document with the highest cross-encoder score.
3. **Augment:** Create your RAG string exactly formatted as: `"Context: [best_document] Question: [prompt]"`.
4. **Predict:** Pass this augmented string to the `facebook/bart-large-mnli` zero-shot classifier, using the 5 options (A, B, C, D, E) as your `candidate_labels`.
5. **Score:** Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., `['C', 'A', 'E']`) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [ ]:
def map_at_3(predicted_letters, correct_letter):
    for rank, letter in enumerate(predicted_letters[:3], 1):
        if letter == correct_letter:
            return 1.0 / rank
    return 0.0

option_cols = ['A', 'B', 'C', 'D', 'E']
map3_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    options = {col: str(row[col]) for col in option_cols}
    correct_letter = row['answer']

    # Step 1: Retrieve top k=5
    prompt_emb = model.encode([prompt]).astype('float32')
    _, indices_5 = index.search(prompt_emb, 5)
    docs_5 = [kb[j] for j in indices_5[0]]

    # Step 2: Rerank — pick best doc
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs_5[int(np.argmax(ce_scores))]

    # Step 3: Build RAG string
    rag_string = f"Context: {best_doc} Question: {prompt}"

    # Step 4: Zero-shot classify
    candidate_labels = list(options.values())
    result = zs(rag_string, candidate_labels=candidate_labels)

    # Step 5: Map top 3 predicted texts back to letters
    text_to_letter = {v: k for k, v in options.items()}
    top3_letters = [text_to_letter[label] for label in result['labels'][:3]]

    score = map_at_3(top3_letters, correct_letter)
    map3_scores.append(score)

    print(f"Row {i:2d} | Correct: {correct_letter} | Top3: {top3_letters} | MAP@3: {score:.4f}")

avg_map3 = np.mean(map3_scores)
print(f"Average MAP@3 over 20 rows = {round(avg_map3, 3)}")

Row  0 | Correct: B | Top3: ['B', 'D', 'A'] | MAP@3: 1.0000
Row  1 | Correct: A | Top3: ['A', 'E', 'C'] | MAP@3: 1.0000
Row  2 | Correct: C | Top3: ['C', 'D', 'B'] | MAP@3: 1.0000
Row  3 | Correct: B | Top3: ['B', 'D', 'A'] | MAP@3: 1.0000
Row  4 | Correct: A | Top3: ['A', 'B', 'C'] | MAP@3: 1.0000
Row  5 | Correct: C | Top3: ['B', 'C', 'A'] | MAP@3: 0.5000
Row  6 | Correct: E | Top3: ['E', 'B', 'D'] | MAP@3: 1.0000
Row  7 | Correct: A | Top3: ['A', 'B', 'C'] | MAP@3: 1.0000
Row  8 | Correct: A | Top3: ['A', 'C', 'D'] | MAP@3: 1.0000
Row  9 | Correct: A | Top3: ['A', 'B', 'C'] | MAP@3: 1.0000
Row 10 | Correct: C | Top3: ['C', 'A', 'D'] | MAP@3: 1.0000
Row 11 | Correct: B | Top3: ['B', 'C', 'A'] | MAP@3: 1.0000
Row 12 | Correct: D | Top3: ['D', 'A', 'B'] | MAP@3: 1.0000
Row 13 | Correct: E | Top3: ['E', 'D', 'B'] | MAP@3: 1.0000
Row 14 | Correct: E | Top3: ['E', 'A', 'D'] | MAP@3: 1.0000
Row 15 | Correct: E | Top3: ['E', 'B', 'C'] | MAP@3: 1.0000
Row 16 | Correct: C | Top3: ['C', 'B', '